In [241]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt
import streamlit as st
import pickle

In [242]:
df = pd.read_csv('Churn_Modelling.csv')

In [243]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [244]:
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [245]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 966.1 KB


In [246]:
df.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [247]:
label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])

In [248]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [249]:
ohe = OneHotEncoder(sparse=False)
country_encoded = ohe.fit_transform(df[['Geography']])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [250]:
country_encoded_df = pd.DataFrame(country_encoded, columns=ohe.get_feature_names_out(['Geography']))

In [251]:
df = pd.concat([df, country_encoded_df], axis=1)

In [252]:
df.drop('Geography', axis=1, inplace=True)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [253]:
# Make sure ohe object exists from your OneHotEncoder cell
print("ohe object:", ohe)
print("ohe categories:", ohe.categories_)

# Save it
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(ohe, file)

print("Saved successfully")

# Verify
import os
print(f"File size: {os.path.getsize('onehot_encoder_geo.pkl')} bytes")

ohe object: OneHotEncoder(sparse=False, sparse_output=False)
ohe categories: [array(['France', 'Germany', 'Spain'], dtype=object)]
Saved successfully
File size: 585 bytes


In [254]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(ohe,file)


In [255]:
X = df.drop('Exited', axis=1)
y = df['Exited']

In [256]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [257]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)
    # to save the scaler object for later use in the Streamlit app

In [258]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [259]:
(X_train.shape[1],) #input neurons

(12,)

In [260]:
model = Sequential([
    Dense(80, activation='relu', input_shape=(X_train.shape[1],)), #Hidden layer 1
    Dense(40, activation='relu'), #Hidden layer 2
    Dense(1, activation='sigmoid') #Output layer
])

/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [261]:
model.summary()
#params are the combination of weights and biases in the model. The number of parameters in each layer is calculated as follows:
#For the first hidden layer:
#Number of parameters = (number of input features * number of neurons) + number of neurons
#= (12 * 70) + 70 = 770
#For the second hidden layer:
#Number of parameters = (number of neurons in the previous layer * number of neurons) + number of neurons
#= (70 * 35) + 35 = 2485
#For the output layer:
#Number of parameters = (number of neurons in the previous layer * number of output neurons) + number of output neurons
#= (35 * 1) + 1 = 36
#Total parameters = 770 + 2485 + 36 = 3291

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 80)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 40)             │         3,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │            41 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,321 (16.88 KB)

 Trainable params: 4,321 (16.88 KB)

 Non-trainable params: 0 (0.00 B)

In [262]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01) #we can use any optimizer like SGD, RMSprop, etc. but Adam is generally a good choice for most problems
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x304ea6ac0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [263]:
## compile the model
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])
#for multicalss we can use sparse_categorical_crossentropy

In [264]:
## Set up the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [265]:
## Set up the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [266]:
## Set up Early Stopping, when the model stops improving on the validation set for a certain number of epochs (patience), it will stop training and restore the best weights, when the loss is not decreasing for 7 consecutive epochs, it will stop training and restore the best weights
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=7,restore_best_weights=True)


In [267]:
### Train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=90,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8378 - loss: 0.3975 - val_accuracy: 0.8520 - val_loss: 0.3698
Epoch 2/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8553 - loss: 0.3554 - val_accuracy: 0.8625 - val_loss: 0.3452
Epoch 3/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8543 - loss: 0.3502 - val_accuracy: 0.8605 - val_loss: 0.3419
Epoch 4/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8591 - loss: 0.3430 - val_accuracy: 0.8595 - val_loss: 0.3413
Epoch 5/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8610 - loss: 0.3425 - val_accuracy: 0.8580 - val_loss: 0.3542
Epoch 6/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8619 - loss: 0.3400 - val_accuracy: 0.8560 - val_loss: 0.3378
Epoch 7/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8640 - loss: 0.3341 - val_accuracy: 0.8575 - val_loss: 0.3390
Epoch 8/90
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8629 - loss: 0.3357 - val_accuracy: 0.

In [268]:
model.save('model.h5')

In [269]:
# ...existing code...
%pip install -U "protobuf==3.20.3"
# ...existing code...

7254.06s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
Note: you may need to restart the kernel to use updated packages.


In [270]:
# ...existing code...
import sys, pkgutil
print("Python:", sys.executable)
print("tensorflow installed:", pkgutil.find_loader("tensorflow") is not None)
print("tensorboard installed:", pkgutil.find_loader("tensorboard") is not None)
# ...existing code...

Python: /opt/anaconda3/bin/python
tensorflow installed: True
tensorboard installed: True


In [271]:
# ...existing code...
%load_ext tensorboard
%tensorboard --logdir logs/fit --port 6010
# ...existing code...

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6010 (pid 63500), started 0:51:54 ago. (Use '!kill 63500' to kill it.)

In [272]:
##load pickle files
